# Laboratory 01 — Thermal equilibrium

In this laboratory two bodies exchange energy quanta at random, one at a time, until they
settle on a shared temperature — and you will measure how fast, and how reliably, that
happens.

Work through it in order. Where the notebook asks you to predict, write your prediction in
the cell provided **before** running the next cell. That is not a ritual: a prediction you
have committed to is the only reliable way to discover that you were wrong.

## Model specification

| | |
|---|---|
| **System** | two bodies A and B, modelled as Einstein solids with $N_A$ and $N_B$ oscillators |
| **Dynamics** | one randomly chosen quantum hops per step, weighted toward leaving the fuller body |
| **Boundary** | closed and isolated as a pair; total quanta (total energy) exactly conserved |
| **Ensemble** | microcanonical for the joint system, exactly as in module 8 |
| **Ignored** | the physical attempt rate (time is counted in steps, not seconds), spatial structure within a body |
| **Valid when** | both bodies sit in the classical, high-temperature equipartition regime |
| **Failure modes** | low temperature, unequal quantum sizes between the bodies, too few oscillators |

All the physics lives in `thermolab.equilibrium` — open it and read it. Nothing in this
course is hidden inside a framework.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import equilibrium
from thermolab.constants import K_B
from thermolab.validation import scaling_exponent, seed_study

QUANTUM = 20.0 * K_B  # an arbitrary but fixed energy quantum, shared by both bodies

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"k_B = {K_B:.6e} J/K")
print(f"quantum = {QUANTUM:.6e} J")

## Part 1 — Watch two bodies find a common temperature

Start with two modestly sized bodies at very different temperatures.

In [ ]:
state = equilibrium.from_temperatures(150, 50, 500.0, 250.0, QUANTUM)
n_steps = 20000
result = equilibrium.simulate_energy_exchange(state, n_steps, rng)

t_eq = equilibrium.equilibrium_temperature(
    state.heat_capacity_a, state.temperature_a, state.heat_capacity_b, state.temperature_b
)
predicted_gap = equilibrium.predicted_relaxation(state, result.steps)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(result.steps, result.temperature_a, lw=0.8, color="crimson", label="body A")
ax.plot(result.steps, result.temperature_b, lw=0.8, color="steelblue", label="body B")
ax.plot(result.steps, t_eq + predicted_gap, lw=1.4, ls="--", color="black",
        label="predicted (analytic)")
ax.axhline(t_eq, color="grey", ls=":", lw=1.0, label="T_eq")
ax.set_xlabel("step")
ax.set_ylabel("temperature (K)")
ax.set_title(f"N_A = {state.n_a}, N_B = {state.n_b}: two bodies relaxing to T_eq")
ax.legend()
plt.tight_layout()
plt.show()

print(f"T_eq predicted            {t_eq:.3f} K")
print(f"T_A after {n_steps} steps   {result.temperature_a[-1]:.3f} K")
print(f"T_B after {n_steps} steps   {result.temperature_b[-1]:.3f} K")
print(f"relaxation time tau = {equilibrium.relaxation_time(state):.1f} steps")

The gap between the two curves shrinks smoothly, without ever changing sign.

### Predict

Before running the next cell, write down what you expect if body B is made ten times
*smaller* than in this run (so the two bodies are very different sizes) while the starting
temperatures stay the same. Does the equilibrium temperature move toward $T_A(0)$, toward
$T_B(0)$, or stay at the midpoint? Be specific.

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)

for ax, (n_a, n_b) in zip(axes, ((30, 10), (300, 100), (3000, 1000)), strict=True):
    s = equilibrium.from_temperatures(n_a, n_b, 500.0, 250.0, QUANTUM)
    tau = equilibrium.relaxation_time(s)
    steps_needed = int(6 * tau)
    run = equilibrium.simulate_energy_exchange(s, steps_needed, rng)
    gap = run.temperature_a - run.temperature_b
    delta_t0 = s.temperature_a - s.temperature_b

    ax.plot(run.steps / tau, gap / delta_t0, lw=1.0, color="steelblue")
    ax.plot(run.steps / tau, equilibrium.predicted_relaxation(s, run.steps) / delta_t0,
            lw=1.2, ls="--", color="crimson")
    ax.set_xlabel("step / tau")
    ax.set_title(f"N = {n_a + n_b}")

axes[0].set_ylabel(r"$(T_A - T_B) \,/\, (T_{A,0} - T_{B,0})$")
plt.tight_layout()
plt.show()

Every panel starts at a fractional gap of exactly $1$ and decays along the same dashed
curve, whatever the system size. What changes is only the scatter around that curve —
smaller systems wander further from the smooth prediction, for the same reason a smaller
gas sample's pressure wobbles more.

## Part 2 — Is the equilibrium temperature really the weighted average?

Measure, do not assume. Vary the two bodies' relative sizes and compare the long-run
temperature with the boxed formula $T_{eq} = (C_A T_{A,0} + C_B T_{B,0})/(C_A + C_B)$.

In [ ]:
def measure_equilibrium_temperature(n_a, n_b, t_a0, t_b0, rng, n_tau=8):
    s = equilibrium.from_temperatures(n_a, n_b, t_a0, t_b0, QUANTUM)
    n_steps = int(n_tau * equilibrium.relaxation_time(s))
    run = equilibrium.simulate_energy_exchange(s, n_steps, rng)
    return float(run.temperature_a[-1]), s


configs = [(150, 150, 500.0, 250.0), (150, 50, 500.0, 250.0), (50, 150, 500.0, 250.0)]
for n_a, n_b, t_a0, t_b0 in configs:
    measured, s = measure_equilibrium_temperature(n_a, n_b, t_a0, t_b0, rng)
    predicted = equilibrium.equilibrium_temperature(
        s.heat_capacity_a, s.temperature_a, s.heat_capacity_b, s.temperature_b
    )
    print(f"N_A={n_a:4d} N_B={n_b:4d}   measured {measured:7.2f} K   predicted {predicted:7.2f} K")

When the two bodies are the same size the equilibrium sits at the midpoint, $375\ K$. Make
body A three times larger than body B (or vice versa) and the equilibrium temperature shifts
toward whichever body has the larger heat capacity — exactly as the weighted-average
formula predicts, and nothing like a plain average of the two starting temperatures.

## Part 3 — How predictable is the approach to equilibrium?

The relaxation *curve* is the same at every size (Part 1). What changes with size is how
tightly individual runs cluster around it — the same $N$-dependent steadying met
throughout this course.

In [ ]:
def relative_spread_at_one_tau(n_a, n_b, n_samples, rng):
    s = equilibrium.from_temperatures(n_a, n_b, 500.0, 250.0, QUANTUM)
    tau = equilibrium.relaxation_time(s)
    checkpoint = max(int(round(tau)), 1)
    gaps = []
    for _ in range(n_samples):
        run = equilibrium.simulate_energy_exchange(s, checkpoint, rng)
        gaps.append(float(run.temperature_a[-1] - run.temperature_b[-1]))
    gaps = np.array(gaps)
    return float(gaps.std(ddof=1) / abs(gaps.mean())), s.total_quanta


sizes = [(15, 5), (50, 17), (150, 50), (500, 167), (1500, 500)]
spreads, totals = [], []
for n_a, n_b in sizes:
    spread, q_total = relative_spread_at_one_tau(n_a, n_b, 30, rng)
    spreads.append(spread)
    totals.append(q_total)
    print(f"N_A={n_a:5d} N_B={n_b:5d}   Q={q_total:6d}   relative spread = {spread:.4f}")

exponent = scaling_exponent(totals, spreads)
print(f"\nfitted exponent = {exponent:.3f}   (theory: roughly -0.5)")

plt.figure(figsize=(6, 4.5))
plt.loglog(totals, spreads, "o", label="measured")
plt.loglog(totals, spreads[0] * (np.array(totals) / totals[0]) ** -0.5, "-", label=r"$Q^{-1/2}$")
plt.xlabel("Q (total quanta)")
plt.ylabel("relative spread of the gap, one tau in")
plt.title(f"fitted slope {exponent:.2f}")
plt.legend()
plt.tight_layout()
plt.show()

## Part 4 — Automated checks

A simulation you have not checked is a picture, not evidence. These are the same assertions
that run in the project's test suite.

In [ ]:
check_state = equilibrium.from_temperatures(150, 50, 500.0, 250.0, QUANTUM)
check_steps = int(8 * equilibrium.relaxation_time(check_state))
check_run = equilibrium.simulate_energy_exchange(check_state, check_steps, np.random.default_rng(1))

# 1. Energy conservation -- every step only relabels which body owns one quantum.
assert np.all(check_run.q_a + check_run.q_b == check_state.total_quanta)

# 2. The equilibrium temperature, across independent seeds and with an honest error bar.
target = equilibrium.equilibrium_temperature(
    check_state.heat_capacity_a, check_state.temperature_a,
    check_state.heat_capacity_b, check_state.temperature_b,
)


def final_temperature_a(r):
    return equilibrium.simulate_energy_exchange(check_state, check_steps, r).temperature_a[-1]


study = seed_study(final_temperature_a, n_seeds=12)
assert study.agrees_with(target, n_sigma=3.5)

# 3. The relaxation curve, one relaxation time in.
checkpoint = int(equilibrium.relaxation_time(check_state))
predicted = float(equilibrium.predicted_relaxation(check_state, np.array([checkpoint]))[0])


def gap_after_one_tau(r):
    run = equilibrium.simulate_energy_exchange(check_state, checkpoint, r)
    return float(run.temperature_a[-1] - run.temperature_b[-1])


gap_study = seed_study(gap_after_one_tau, n_seeds=16)
assert gap_study.agrees_with(predicted, n_sigma=3.5)

print(f"T_eq         measured {study.mean:.3f} +/- {study.standard_error:.3f}   vs   {target:.3f}")
print(
    f"gap at tau   measured {gap_study.mean:.3f} +/- {gap_study.standard_error:.3f}   "
    f"vs   {predicted:.3f}"
)
print("\nall checks passed")

## Part 5 — Explore it yourself

The sliders below let you vary the two bodies' sizes and starting temperatures freely. Two
experiments worth doing:

1. Make the two bodies very unequal in size and watch how little the larger one's temperature
   moves — this is why a thermometer (small) barely disturbs the thing it measures (large).
2. Find a size so small that the relaxation curve looks visibly noisy rather than smooth, and
   say what standard you used to decide "noisy".

In [ ]:
import ipywidgets as widgets


def explore(n_a=150, n_b=50, t_a0=500.0, t_b0=250.0, n_tau=6.0):
    s = equilibrium.from_temperatures(n_a, n_b, t_a0, t_b0, QUANTUM)
    tau = equilibrium.relaxation_time(s)
    n_steps = max(int(n_tau * tau), 10)
    run = equilibrium.simulate_energy_exchange(s, n_steps, np.random.default_rng(0))
    t_eq = equilibrium.equilibrium_temperature(
        s.heat_capacity_a, s.temperature_a, s.heat_capacity_b, s.temperature_b
    )

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(run.steps, run.temperature_a, lw=1.0, color="crimson", label="body A")
    ax.plot(run.steps, run.temperature_b, lw=1.0, color="steelblue", label="body B")
    ax.axhline(t_eq, color="grey", ls=":", lw=1.0, label="T_eq")
    ax.set_xlabel("step")
    ax.set_ylabel("temperature (K)")
    ax.set_title(f"tau = {tau:.0f} steps, T_eq = {t_eq:.1f} K")
    ax.legend()
    plt.tight_layout()
    plt.show()


widgets.interact_manual(
    explore,
    n_a=widgets.IntSlider(min=5, max=3000, step=5, value=150, description="N_A"),
    n_b=widgets.IntSlider(min=5, max=3000, step=5, value=50, description="N_B"),
    t_a0=widgets.FloatSlider(min=100, max=900, step=10, value=500.0, description="T_A0 (K)"),
    t_b0=widgets.FloatSlider(min=100, max=900, step=10, value=250.0, description="T_B0 (K)"),
    n_tau=widgets.FloatSlider(min=1, max=10, step=0.5, value=6.0, description="steps (x tau)"),
);

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "01-equilibrium.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict in Part 1 that turned out to be wrong, and what specifically was the
   flaw in your reasoning?
2. This model exchanges identical quanta between two idealised Einstein solids. Name one
   conclusion from today that is therefore **not** established about real thermal contact,
   even though the numbers agreed with the derivation.
3. Explain, without equations, why a larger heat capacity pulls the equilibrium temperature
   toward its own starting value.

**Your answers:**

1.
2.
3.